In [1]:
%pip install statsmodels

  Using cached statsmodels-0.14.6-cp314-cp314-win_amd64.whl.metadata (9.8 kB)
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
Using cached statsmodels-0.14.6-cp314-cp314-win_amd64.whl (9.6 MB)
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)

   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [statsmodels]
   -------------------- ------------------- 1/2 [st

In [1]:
import numpy as np
import torch

# Allow numpy arrays in torch.load (PyTorch 2.6 fix)
import torch.serialization
torch.serialization.add_safe_globals([np.core.multiarray._reconstruct])

# Load correct files
y_true = torch.load("y_true.pth", weights_only=False)

y_pred_dense = torch.load("multi_cancer_densenet121_preds.pth", weights_only=False)
y_pred_eff   = torch.load("multi_cancer_effcientnet_preds.pth", weights_only=False)
y_pred_mobile= torch.load("multi_cancer_mobilenetv3_preds.pth", weights_only=False)
y_pred_resnet= torch.load("multi_cancer_resnet18_preds.pth", weights_only=False)

# Convert to numpy
y_true = np.array(y_true)
y_pred_dense = np.array(y_pred_dense)
y_pred_eff = np.array(y_pred_eff)
y_pred_mobile = np.array(y_pred_mobile)
y_pred_resnet = np.array(y_pred_resnet)

C:\Users\RoupB\AppData\Local\Temp\ipykernel_872\2686172243.py:6: DeprecationWarning: numpy.core is deprecated and has been renamed to numpy._core. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.multiarray.
  torch.serialization.add_safe_globals([np.core.multiarray._reconstruct])


In [2]:
import numpy as np
import torch
from statsmodels.stats.contingency_tables import mcnemar, cochrans_q

In [ ]:
# HANDLE PROBABILITIES

def to_labels(pred):
    # If predictions are probabilities/logits
    if len(pred.shape) > 1:
        return np.argmax(pred, axis=1)
    return pred

y_pred_dense = to_labels(y_pred_dense)
y_pred_eff = to_labels(y_pred_eff)
y_pred_mobile = to_labels(y_pred_mobile)
y_pred_resnet = to_labels(y_pred_resnet)

In [ ]:
# SAFETY CHECK

assert len(y_true) == len(y_pred_dense) == len(y_pred_eff) == len(y_pred_mobile) == len(y_pred_resnet), "Mismatch in lengths!"


In [ ]:
# CONVERT TO CORRECT / INCORRECT

dense_correct = (y_pred_dense == y_true).astype(int)
eff_correct = (y_pred_eff == y_true).astype(int)
mobile_correct = (y_pred_mobile == y_true).astype(int)
resnet_correct = (y_pred_resnet == y_true).astype(int)

In [ ]:
# COCHRAN’S Q TEST

from statsmodels.stats.contingency_tables import cochrans_q

data = np.vstack([
    dense_correct,
    eff_correct,
    mobile_correct,
    resnet_correct
]).T

# Run test
result = cochrans_q(data)

# Extract values correctly
q_stat = result.statistic
p_value = result.pvalue

print("\n🔹 Cochran’s Q Test")
print("Q-statistic:", q_stat)
print("p-value:", p_value)


🔹 Cochran’s Q Test
Q-statistic: 2574.211758301579
p-value: 0.0


In [ ]:
# MCNEMAR TEST

def run_mcnemar(model1, model2, name1, name2):
    table = np.zeros((2, 2))

    for i in range(len(y_true)):
        table[model1[i], model2[i]] += 1

    result = mcnemar(table, exact=True)

    print(f"\n🔸 {name1} vs {name2}")
    print("Table:\n", table)
    print("p-value:", result.pvalue)

In [8]:
# Pairwise tests
run_mcnemar(dense_correct, eff_correct, "DenseNet", "EfficientNet")
run_mcnemar(dense_correct, mobile_correct, "DenseNet", "MobileNet")
run_mcnemar(dense_correct, resnet_correct, "DenseNet", "ResNet")
run_mcnemar(eff_correct, mobile_correct, "EfficientNet", "MobileNet")
run_mcnemar(eff_correct, resnet_correct, "EfficientNet", "ResNet")
run_mcnemar(mobile_correct, resnet_correct, "MobileNet", "ResNet")



🔸 DenseNet vs EfficientNet
Table:
 [[ 150. 1371.]
 [  96.  633.]]
p-value: 2.099822809424746e-289

🔸 DenseNet vs MobileNet
Table:
 [[ 164. 1357.]
 [ 125.  604.]]
p-value: 9.124405753723547e-262

🔸 DenseNet vs ResNet
Table:
 [[ 139. 1382.]
 [ 124.  605.]]
p-value: 3.991929983108391e-269

🔸 EfficientNet vs MobileNet
Table:
 [[  90.  156.]
 [ 199. 1805.]]
p-value: 0.025666910882391095

🔸 EfficientNet vs ResNet
Table:
 [[  92.  154.]
 [ 171. 1833.]]
p-value: 0.3748253580707024

🔸 MobileNet vs ResNet
Table:
 [[  88.  201.]
 [ 175. 1786.]]
p-value: 0.19723642137584366


Based on the sigfiniace level Desnset performs the best out of all four models.
